In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [7]:
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/MAPLES-DR/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
TOPO = PATH / "3-topo"
IMG = sorted(list(RAW.glob("*.png")))[18].stem

fundus_gt = FundusData(image=RAW / (IMG + ".png"), av=AV / (IMG + ".png"))
trees_gt = VTree.load(TOPO / f"{IMG}_art.npz"), VTree.load(TOPO / f"{IMG}_vei.npz")
topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=5),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=5),
)


od_mac = segment(open_image(RAW / (IMG + ".png"))).numpy(force=True).argmax(axis=0)
fundus_gt = fundus_gt.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")
fundus = fundus_gt.copy()
_ = segment_av(fundus)

av2tree = GNNAVSegToTree()
graph = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()

print(IMG)

20051110_35926_0400_PP


In [ ]:
digraph = VBranchDigraph.from_graph(graph, max_distance=200)
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])
solved_tree = digraph.optimize_tree(keep_invalid_branch=True)

m = Mosaic(
    3, cols_titles=["Predicted", "Predicted with GT Topology", "Ground Truth"], cell_height=600, background=fundus.image
)
fundus.draw(view=m[0])
draw_graph(graph, view=m[0], edge_labels=True, node_labels=True)
fundus.draw(view=m[1])
draw_tree(
    solved_tree,
    view=m[1],
    branch_color="subtree",
    edge_labels=True,
    node_labels=True,
    bspline_dir=True,
)
fundus_gt.draw(view=m[2])
draw_trees(trees_gt, view=m[2])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [ ]:
read_branch_topology(digraph.graph, topo_gt[1])

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TopologicalLabel, read_branch_topology


(
    TopologicalLabel(read_branch_topology(digraph.graph, topo_gt[0])[0][227]),
    TopologicalLabel(read_branch_topology(digraph.graph, topo_gt[0])[0][102]),
)

(TopologicalLabel(subtree=4, rank=5, branching_pattern=[True, True, True, True, True]),
 TopologicalLabel(subtree=4, rank=4, branching_pattern=[True, True, True, True]))

In [ ]:
solved_tree.subtrees_branch_labels()[[66, 272]]

array([17,  5])

In [18]:
pd.DataFrame(digraph.lines_by_branch(7))

,0,1,2,3,4,5,6
0,166.0,126.0,7.0,126.0,1.0,0.999958,0.997527
1,-1.0,259.0,7.0,126.0,1.0,0.002473,0.997527
2,7.0,95.0,183.0,95.0,1.0,0.997527,0.997527
3,7.0,95.0,8.0,95.0,1.0,0.997527,0.997527
4,192.0,35.0,7.0,126.0,0.0,0.999447,0.997527
5,7.0,95.0,146.0,45.0,0.0,0.997527,0.997527
6,7.0,95.0,136.0,45.0,0.0,0.997527,0.995308
7,7.0,95.0,154.0,190.0,0.0,0.997527,0.992905
8,7.0,126.0,166.0,35.0,0.0,0.002473,0.999958
9,7.0,126.0,192.0,69.0,0.0,0.002473,0.999447


In [7]:
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

In [8]:
%timeit digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

79.4 ms ± 870 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
VBranchDigraph.from_graph(graph, max_distance=300)

In [10]:
def draw_cone(branch_id: int, first_tip: bool, view=None, pos_tolerance=15, max_dist=100, max_angle=30):
    sqr_max_dist = max_dist * max_dist
    sqr_pos_tolerance = pos_tolerance * pos_tolerance

    min_cos = np.cos(np.deg2rad(max_angle))

    geodata = graph.geometric_data()
    tips_pos = (
        geodata.tip_coord(branch_id=branch_id, first_tip=first_tip).astype(np.float64).reshape(-1, 2)
    )  # [branch_id x (tip0, tip1), (y,x)]
    tips_tan = geodata.tip_tangent(branch_id=branch_id, first_tip=first_tip).reshape(
        -1, 2
    )  # [branch_id x (tip0, tip1), (y,x)]

    yy, xx = np.meshgrid(np.arange(fundus.image.shape[1]), np.arange(fundus.image.shape[2]), indexing="ij")
    yx = np.stack((yy, xx), axis=-1).reshape(-1, 2)

    tips_dtan = tips_pos[:, None, :] - yx[None, :, :]  # (tip_origin, tip_destination, yx)
    tips_dsqr = np.square(tips_dtan).sum(axis=2)
    tips_dtan /= np.sqrt(tips_dsqr)[..., None] + 1e-8

    # === VICINITY CHECK ===
    # Given a tip p0 with tangent t0 (oriented towards its curve)
    # we define a cone oriented towards -t0 with apex at p0 + t0 * pos_tolerance (so the tip itself is inside the cone)
    # and opening angle max_angle at distance pos_tolerance and 60 degrees at distance 0 from the apex.
    apex = tips_pos + tips_tan * pos_tolerance  # Cone apex position
    apex2tips = apex[:, None, :] - yx[None, :, :]
    apex2tips_dsqr = np.square(apex2tips).sum(axis=2)
    apex2tips /= np.sqrt(apex2tips_dsqr)[..., None] + 1e-8
    apex_cos = (tips_tan[:, None, :] * apex2tips).sum(axis=2)
    inside_cone = ((apex_cos >= min_cos) | (tips_dsqr <= sqr_pos_tolerance)) & (tips_dsqr <= sqr_max_dist)
    yx = yx[inside_cone[0]]
    map = np.zeros(fundus.image.shape[1:], dtype=np.uint8)
    map[yx[:, 0], yx[:, 1]] = 1
    if view is not None:
        view.add_label(map, "cone", opacity=0.2)